# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamzaabutt/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
# Code check for Section 1: Verify distribution of CTR across content pages
import pandas as pd
import duckdb

con = duckdb.connect()
rel = 'hf://datasets/FlyRank/internship-warehouse'

# Load sample dataset to inspect overall CTR stats
df_stats = con.sql(f"""
    SELECT
        AVG(clicks * 1.0 / NULLIF(impressions, 0)) AS avg_ctr,
        MEDIAN(clicks * 1.0 / NULLIF(impressions, 0)) AS median_ctr
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    WHERE impressions > 50
""").df()

print("Dataset Baseline CTR Stats:")
print(df_stats)

Binary Classification (with secondary Probability Scoring)

Why this type?
Content optimization teams need to make discrete, actionable decisions: either allocate editorial resources to rewrite a page or leave it alone. Binary classification cleanly frames this decision by separating high-opportunity pages from stable or low-yield ones. Outputting a probability score alongside the classification allows ranking candidate pages by priority, ensuring high-impact interventions happen first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
df_target_check = con.sql(f"""
    WITH page_aggregates AS (
        SELECT
            content_id,
            SUM(impressions) AS total_impressions,
            SUM(clicks) AS total_clicks,
            SUM(clicks) * 1.0 / NULLIF(SUM(impressions), 0) AS aggregate_ctr
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
        GROUP BY content_id
        HAVING SUM(impressions) >= 100
    )
    SELECT
        content_id,
        total_impressions,
        aggregate_ctr,
        CASE
            WHEN total_impressions > 1000 AND aggregate_ctr < 0.02 THEN 1
            ELSE 0
        END AS is_high_opportunity
    FROM page_aggregates
    LIMIT 5
""").df()

print("Sample Target Framing Dataframe:")
print(df_target_check)

Target Variable: is_high_opportunity (Binary: 1 or 0)

Label Definition & Source:

This is a defined rule (proxy label) based on observed outcomes over a 30-day window:

Label = 1 (High Opportunity): A page with high impression volume (top 25th percentile) but an observed CTR significantly below the baseline median for its impression cohort.

Label = 0 (Low Opportunity / Stable): Pages with low impressions or pages that already convert impressions into clicks at or above expectation.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

PR-AUC (Precision-Recall Area Under Curve), evaluated alongside Precision @ K (e.g., Precision @ 50).

What number means 'good'?

Precision @ 50 >= 0.75: Out of the top 50 pages recommended for editorial rewrites each week, at least 38 (75%) must successfully achieve a statistically significant CTR lift post-refresh.

PR-AUC >= 0.65: Because high-opportunity pages represent a class imbalance (typically <10% of total URLs), PR-AUC is far more informative than ROC-AUC or standard accuracy, ensuring high precision on positive recommendations without false alarm fatigue.

In [ ]:
class_balance = con.sql(f"""
    WITH page_aggregates AS (
        SELECT
            content_id,
            SUM(impressions) AS total_impressions,
            SUM(clicks) * 1.0 / NULLIF(SUM(impressions), 0) AS aggregate_ctr
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
        GROUP BY content_id
        HAVING SUM(impressions) >= 100
    )
    SELECT
        AVG(CASE WHEN total_impressions > 1000 AND aggregate_ctr < 0.02 THEN 1 ELSE 0 END) AS positive_class_ratio
    FROM page_aggregates
""").df()

print(f"Positive Class Ratio (Class Imbalance): {class_balance['positive_class_ratio'].iloc[0]:.2%}")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
df_unit_of_analysis = con.sql(f"""
    SELECT
        content_id,
        COUNT(DISTINCT date) AS active_days,
        SUM(impressions) AS rolling_30d_impressions,
        SUM(clicks) AS rolling_30d_clicks,
        ROUND(AVG(position), 2) AS avg_position,
        ROUND(SUM(clicks) * 100.0 / NULLIF(SUM(impressions), 0), 2) AS ctr_percentage
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    GROUP BY content_id
    LIMIT 5
""").df()

# Display the dataframe structure
print("Unit of Analysis DataFrame (1 Row = 1 Unique URL / Content ID):")
df_unit_of_analysis

Unit of Analysis: One row = One unique Content URL aggregated over a rolling 30-day window.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
position_ctr_breakdown = con.sql(f"""
    SELECT
        ROUND(position) AS rounded_position,
        COUNT(*) AS total_records,
        ROUND(AVG(clicks * 100.0 / NULLIF(impressions, 0)), 2) AS avg_ctr_pct
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    WHERE position BETWEEN 1 AND 10 AND impressions > 50
    GROUP BY 1
    ORDER BY 1
""").df()

print("Non-Linear CTR Decay across SERP Positions (Proves why fixed rules break):")
print(position_ctr_breakdown)

A fixed rule (e.g., IF impressions > 1000 AND ctr < 2% THEN rewrite) fails because search performance depends on multi-variable nonlinear interactions:

Position Bias: A CTR of 1.5% is terrible for Position 1, but exceptional for Position 9. Rules struggle to continuously map dynamic expected CTR across average positions.

Intent & Query Volatility: Seasonal trends and changing search intent mean fixed thresholds quickly become obsolete.

Multi-Feature Interaction: Machine learning models capture high-dimensional feature interactions—combining position decay, historical impression growth rates, content length, and category baselines simultaneously—to identify non-obvious underperformers that fixed rules miss.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.